# 📈 Statistical Analysis — Tomato Market Prices
Three focused analyses that give numbers to what the charts show visually.

1. **Seasonal Decomposition** — separates trend, seasonality, and noise
2. **Price Regression** — does arrival quantity actually predict price?
3. **Summary Stats Table** — one clean reference table for the report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

SHORT = {
    'Mumbai Apmc'            : 'Mumbai',
    'Nagpur Apmc'            : 'Nagpur',
    'Nasik Apmc'             : 'Nasik',
    'Pimpalgaon Baswant Apmc': 'Pimpalgaon',
    'Pune Apmc'              : 'Pune',
    'Pune(Manjri) Apmc'      : 'Pune Manjri',
}

df = pd.read_csv('../data/master_df.csv')
df['date']         = pd.to_datetime(df['date'])
df['month']        = df['date'].dt.to_period('M')
df['month_num']    = df['date'].dt.month
df['year']         = df['date'].dt.year
df['market_short'] = df['market'].map(SHORT)

print("✅ Ready")

---
## Analysis 1 — Seasonal Decomposition

**What:** Splits the price time series into 3 components:
- **Trend** — the overall direction prices are moving (up/down/flat)
- **Seasonal** — the repeating pattern every year (the cycle we already saw)
- **Residual** — what's left after removing trend and season (random noise, shocks)

**Why:** The charts *showed* seasonality visually. This *proves* it mathematically
and tells us exactly how big the seasonal swing is in rupees.

We run this on the overall average price (all markets combined, monthly).

In [ ]:
# Build monthly average series — all markets combined
monthly = (df.groupby('month')['modal_price']
             .mean()
             .reset_index()
             .sort_values('month'))
monthly.index = monthly['month'].dt.to_timestamp()
price_series  = monthly['modal_price']

# Seasonal decomposition
# period=12 → we expect a 12-month (annual) seasonal cycle
# model='additive' → seasonal effect adds/subtracts a fixed amount from trend
decomp = seasonal_decompose(price_series, model='additive', period=6, extrapolate_trend='freq')

# Plot
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.patch.set_facecolor('#FFFFFF')

components = [
    (price_series,      'Observed Price',  '#D62839'),
    (decomp.trend,      'Trend',           '#1B6CA8'),
    (decomp.seasonal,   'Seasonal Effect', '#2D9E6B'),
    (decomp.resid,      'Residual (Noise)','#888888'),
]
for ax, (data, title, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, linewidth=2)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=6)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
    ax.grid(alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if title == 'Seasonal Effect':
        ax.axhline(0, color='black', linewidth=0.8, linestyle=':')

fig.suptitle('Seasonal Decomposition — Maharashtra Tomato Prices\n(Monthly Avg | All Markets)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../charts/13_seasonal_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

# Key numbers
seasonal_amplitude = decomp.seasonal.max() - decomp.seasonal.min()
peak_season_month  = decomp.seasonal.groupby(decomp.seasonal.index.month).mean().idxmax()
low_season_month   = decomp.seasonal.groupby(decomp.seasonal.index.month).mean().idxmin()
month_names        = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print(f"Seasonal amplitude : ₹{seasonal_amplitude:,.0f}/quintal")
print(f"Peak season month  : {month_names[peak_season_month-1]}")
print(f"Low season month   : {month_names[low_season_month-1]}")
print(f"Trend (start)      : ₹{decomp.trend.dropna().iloc[0]:,.0f}")
print(f"Trend (end)        : ₹{decomp.trend.dropna().iloc[-1]:,.0f}")
trend_direction = "upward" if decomp.trend.dropna().iloc[-1] > decomp.trend.dropna().iloc[0] else "downward"
print(f"Trend direction    : {trend_direction} over 20 months")

- We use **period=6** (semi-annual cycle) instead of 12 because our dataset 
  covers only 20 months — seasonal decompose needs at least 2 full cycles 
  to work, so 6-month periods fit within our data window.
  The pattern still captures the two price peaks per year (summer + winter).

### What the Output Tells Us

- **Seasonal amplitude** = the rupee swing caused purely by the time of year.
  If amplitude is ₹800 — that means just being in July vs February 
  adds/subtracts ₹800 to the price, regardless of anything else.
- **Trend** panel shows the underlying direction after removing seasonal noise.
  A flat or downward trend in 2026 confirms the market correction we saw.
- **Residual** panel = everything unexplained. Big spikes here = shock events
  (the ₹10,500 Pune day, monsoon disruptions, sudden demand from other states).
- This is why farmers get hurt — the seasonal swing is massive and predictable,
  but they have no tools to hedge against it.

---
## Analysis 2 — Does Arrival Quantity Predict Price?

**What:** Linear regression — arrival quantity (X) vs modal price (Y), per market.

**Why:** Common sense says more supply = lower price. 
But is that actually true in our data, and how strong is the relationship?

We use `scipy.stats.linregress` — gives us slope, r-value, and p-value.
- **Slope** → for every 1 MT increase in arrivals, price changes by this much
- **r-value** → strength of relationship (-1 to +1)
- **p-value** → is this statistically real or just random? (p < 0.05 = real)

In [ ]:
print("=== REGRESSION: Arrival Quantity → Modal Price (per market) ===\n")
print(f"{'Market':<15} {'Slope':>8} {'R':>7} {'R²':>7} {'P-value':>12} {'Significant?':>13}")
print("-" * 68)

results = []
for mkt in df['market'].unique():
    sub  = df[(df['market']==mkt) & (df['arrival_quantity']>0)].copy()
    # Remove top 1% outliers for cleaner regression
    sub  = sub[sub['arrival_quantity'] <= sub['arrival_quantity'].quantile(0.99)]
    
    slope, intercept, r, p, se = stats.linregress(
        sub['arrival_quantity'], sub['modal_price']
    )
    sig  = "✅ Yes" if p < 0.05 else "❌ No"
    name = SHORT[mkt]
    print(f"{name:<15} {slope:>8.2f} {r:>7.3f} {r**2:>7.3f} {p:>12.4f} {sig:>13}")
    results.append({'market': name, 'slope': slope, 'r': r, 'r2': r**2, 'p': p})

print()
print("Slope = price change (₹) per additional 1 MT of arrivals")
print("R²    = % of price variation explained by arrival quantity alone")

# Visualise regression results
res_df = pd.DataFrame(results).sort_values('r')
MCOLORS_SHORT = {
    'Mumbai':'#D62839','Nagpur':'#1B6CA8','Nasik':'#F4A535',
    'Pimpalgaon':'#2D9E6B','Pune':'#7B2D8B','Pune Manjri':'#E07B39'
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(res_df['market'], res_df['r'],
             color=[MCOLORS_SHORT[m] for m in res_df['market']],
             edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Correlation (r) — Arrival vs Price\nper Market', pad=12)
axes[0].set_xlabel('Pearson r  (negative = more supply → lower price)')

axes[1].barh(res_df['market'], res_df['r2']*100,
             color=[MCOLORS_SHORT[m] for m in res_df['market']],
             edgecolor='white', alpha=0.85)
axes[1].set_title('R² — How Much of Price Variation\nIs Explained by Arrivals Alone?', pad=12)
axes[1].set_xlabel('R² (%)')
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.25, linestyle='--')

plt.suptitle('Regression — Arrival Quantity vs Modal Price',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../charts/14_regression_arrival_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

### What the Output Tells Us

- **Negative slope** = supply-price relationship is real — more arrivals 
  push prices down. Expected economic behaviour.
- **Pimpalgaon has the strongest negative r** — makes total sense.
  It's a production market. When trucks dump 5,000 MT in one day,
  prices genuinely fall. Supply directly sets the price there.
- **Nagpur has weak or near-zero r** — its prices are driven more by 
  *whether trucks arrive at all* (transport, road conditions, weather)
  than by *how many MT* arrive. Distance from source breaks the relationship.
- **Low R²** across all markets tells us arrival quantity alone explains 
  only a small part of price movement. The rest is seasonality, weather, 
  demand shocks, fuel prices, and panic buying/selling. 
  This is why tomato price prediction is genuinely hard.
- **p-value < 0.05** = the relationship is statistically real, not random.

---
## Analysis 3 — Master Summary Table

One clean reference table combining everything.
This feeds directly into the findings report and webpage.

In [ ]:
# Build the master summary table
summary = df.groupby('market_short').agg(
    Records        = ('modal_price',      'count'),
    Min_Price      = ('modal_price',      'min'),
    Max_Price      = ('modal_price',      'max'),
    Mean_Price     = ('modal_price',      'mean'),
    Median_Price   = ('modal_price',      'median'),
    Std_Dev        = ('modal_price',      'std'),
    Total_Arrivals = ('arrival_quantity', 'sum'),
    Avg_Daily_MT   = ('arrival_quantity', 'mean'),
).round(1)

summary['CV_Pct']       = (summary['Std_Dev'] / summary['Mean_Price'] * 100).round(1)
summary['Price_Range']  = (summary['Max_Price'] - summary['Min_Price']).round(0)
summary                 = summary.sort_values('Mean_Price', ascending=False)

print("=== MASTER SUMMARY TABLE ===\n")
print(summary.to_string())

# Save as CSV for the report
summary.to_csv('../report/market_summary.csv')
print("\n✅ Saved to ../report/market_summary.csv")

# Quick ranking printout
print("\n=== MARKET RANKINGS ===")
print(f"\nMost Expensive  : {summary['Mean_Price'].idxmax()} (₹{summary['Mean_Price'].max():,.0f} avg)")
print(f"Cheapest        : {summary['Mean_Price'].idxmin()} (₹{summary['Mean_Price'].min():,.0f} avg)")
print(f"Most Volatile   : {summary['CV_Pct'].idxmax()} (CV = {summary['CV_Pct'].max():.1f}%)")
print(f"Most Stable     : {summary['CV_Pct'].idxmin()} (CV = {summary['CV_Pct'].min():.1f}%)")
print(f"Highest Volume  : {summary['Total_Arrivals'].idxmax()} ({summary['Total_Arrivals'].max():,.0f} MT total)")
print(f"Lowest Volume   : {summary['Total_Arrivals'].idxmin()} ({summary['Total_Arrivals'].min():,.0f} MT total)")
print(f"Widest Range    : {summary['Price_Range'].idxmax()} (₹{summary['Price_Range'].max():,.0f} spread)")

### What the Summary Table Tells Us

This is the single most useful reference in the entire project.
Every number in the findings report and webpage traces back to this table.

Key takeaways in one line each:

- **Nagpur** = highest price, highest risk (buy here only when supply is guaranteed)
- **Nasik** = cheapest, most predictable (best market for bulk procurement)
- **Pimpalgaon** = where Maharashtra's tomato economy actually runs from
- **CV%** = the risk score for traders. Higher CV = harder to plan, 
  more chance of getting caught on wrong side of a price swing
- **Price range** tells you the worst-case scenario per market —
  Nagpur's ₹5,950 spread means a trader could buy at ₹6,150 and 
  next month see prices at ₹200. That's a real business risk.